# ITO5202 : Music Recommendation using Collaborative Filtering

Collaborative filtering (CF) is a technique commonly used to build personalized recommendations on the Web. Some popular websites that make use of the collaborative filtering technology include Amazon, Netflix, iTunes, IMDB, LastFM, Delicious and StumbleUpon. In collaborative filtering, algorithms are used to make automatic predictions about a user's interests by compiling preferences from several users.

In this lab, our task is to use a collaborative algorithm to recommend top artists from the given dataset. The dataset can be downloaded from Moodle. 
<p style="color:red">Complete the required tasks in the tutorial. The activited is denoted as "Task" with the required instructions</p>
<br/>

## Table of Contents

* [ALS Lecture Demo](#als-demo)
* [Use-Case Music Recommendation](#use-case)
    * [Data Loading](#data-loading)
    * [Data Preparation](#data-prep)
    * [Data Exploration](#data-exploration)
    * [Train-Test Split](#train-test-split)
    * [Model Building](#model-building)
    * [Evaluation](#evaluation)
    * [Hyperparameter Tuning and Cross Validation](#cv)
    * [Making Predictions](#predictions)    
* [Lab Tasks](#lab-task-1)
    * [Lab Task 1](#lab-task-1)
    * [Lab Task 2](#lab-task-2)
    * [Lab Task 3](#lab-task-3)
    * [Lab Task 4](#lab-task-4)
    * [Lab Task 5](#lab-task-5)
    * [Lab Task 6](#lab-task-6)
    * [Lab Task 7](#lab-task-7)
    
## Including Libraries and Initializing Spark Context

In [1]:
#import libraries
from pyspark import SparkContext
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession ,Row
from pyspark.sql.functions import col,split
from pyspark.sql.types import StructType,StructField,IntegerType,StringType


appName="Collaborative Filtering with PySpark"
#initialize the spark session
spark = SparkSession.builder.appName(appName).getOrCreate()
#get sparkcontext from the sparksession
sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/13 09:14:52 WARN Utils: Your hostname, Stefans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.115 instead (on interface en0)
26/09/13 09:14:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/envs/ITO5202/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/13 09:14:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Alternating Least Squares DEMO <a class="anchor" name="als-demo"></a>

Please go through the ALS Demo presented in the Lecture to understand the basic flow before starting with the lab tasks.

## Use-Case : Music Recommendation <a class="anchor" name="use-case"></a>
The goal here is to use the data provided to create a recommendation system using collaborative filtering using the social influence data and predict artists a user might like but have not listened to.

> Consider the following example. If user A is a neighbor of user B, and they have similar musical tastes, then there is a very strong tie between them. If user B is a big fan of artist C, and has scrobbled them numerous times, then there is also a strong tie between them. Based on last.fm’s data, user A has not yet listened to artist C (no link has formed between them yet), and there is a good chance that user A will also like artist C.<a href="https://blogs.cornell.edu/info2040/2012/09/20/last-fm-music-reccomendation-incorporating-social-network-ties-and-collaborative-filtering/#:~:text=their%20listening%20frequency.-,Last.,in%20the%20user's%20local%20network." target="_BLANK">Ref</a>


The original dataset is available at <a href="https://www.last.fm/api/" target="_blank">last.fm api</a>. The dataset provided here is a lighter version, resized for the sake of simplicity. The dataset contains three files as follows:
<ul>
    <li><strong>user_artist_data.txt</strong>
        3 columns: <code>userid, artistid, playcount</code></li>
    <li><strong>artist_data.txt</strong>
        2 columns: <code>artistid ,artist_name</code></li>
    <li><strong>artist_alias.txt</strong>
        2 columns: <code>badid, goodid</code>
        [known incorrectly spelt artists and the correct artist id].</li>
</ul>


<a class="anchor" id="lab-task-1"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">1. Lab Task: </strong> 
Import the two other files (user_artist_data.txt and artist_alias.txt) to create two dataframes <code>df_artist_alias</code> and <code>df_user_artist</code>
    
<strong style="color:red">NOTE:</strong> Check the <strong>delimiter</strong> used in these files. <code>\t</code> may not be used for all files.
</div>.




## Data Loading <a class="anchor" name="data-loading"></a>

In [2]:
df = spark.read.text("artist_data.txt")
split_col = split(df['value'], '\t')
df = df.withColumn('artist_id', split_col.getItem(0))
df = df.withColumn('artist_name', split_col.getItem(1))
df_artist=df.drop('value')

In [3]:
# Load user_artist_data.txt to a dataframe called df_user_artist
df_user_artist = (
    spark.read
    .option("sep", " ")
    .option("inferSchema", "true")
    .csv("user_artist_data.txt")
    .toDF("user_id", "artist_id", "playcount")
)


In [10]:
# Load artist_alias.txt to a dataframe called df_artist_alias
df_artist_alias = (
    spark.read
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv("artist_alias.txt")
    .toDF("bad_id", "good_id")
)
print(df_artist_alias)

DataFrame[bad_id: int, good_id: int]


## Data Preparation <a class="anchor" name="data-prep"></a>

The <code>df_user_artist</code> contains <strong>bad ids</strong>, so the <strong>bad ids</strong> in the <code>user_artist_data.txt</code> file need to be remapped to <strong>goodids</strong>. The mapping of <strong>bad_ids</strong> to <strong>good_ids</strong> is in <strong>artist_alias.txt</strong> file. The first task is to create a dictionary of the artist_alias, so that it can be passed over a <strong>broadcast variable</strong>.

Broadcast makes Spark send and hold in memory just one copy for each executor in the cluster. When there are thousands of tasks, and many execute in parallel on each executor, this can save significant network traffic and memory.
But you cannot directly broadcast a dataframe, it has to be converted to a list first

In [5]:
#After loading the artist_alias data to the dataframe, it is converted to a dictionary to be set as a broadcast variable
artist_alias = dict(df_artist_alias.collect())

<a class="anchor" id="lab-task-2"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">2. Lab Task: </strong> 
For the dictionary <strong>artist_alias</strong> which contains key value pair of badid and goodid, create a broadcast variable called <strong>bArtistAlias</strong>.
</div>

In [6]:
bArtistAlias = sc.broadcast(artist_alias)


After the broadcast variable is created, a function to replace the badids by looking up the values from the broadcasted dictionary is implemented for the userArtistRDD.


In [7]:
from pyspark.sql.functions import udf, struct

def lookup_correct_id(artist_id):
    finalArtistID = bArtistAlias.value.get(artist_id)
    if finalArtistID is None:
        finalArtistID = artist_id
    return finalArtistID

lookup_udf = udf(lookup_correct_id, StringType())


/opt/anaconda3/envs/ITO5202/lib/python3.13/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


<a class="anchor" id="lab-task-3"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">3. Lab Task: </strong> 
    Use the udf <code>lookup_udf</code> to replace the "badids" in the <code>df_user_artist</code> dataframe.
</div>

In [8]:
df_user_artist = df_user_artist.withColumn(
    "artist_id",
    lookup_udf(col("artist_id"))
)


When we want to repeteadly access a dataframe or an RDD, it is a good idea to cache them, it helps to speed up applications.

In [9]:
#Uncomment this to use caching
df_user_artist.cache()

DataFrame[user_id: int, artist_id: string, playcount: int]

## Data Exploration <a class="anchor" name="data-exploration"></a>

<a class="anchor" id="lab-task-4"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">4. Lab Task: </strong> 
    Write a query in the function below, to return top <strong>N</strong> artist for a user with user_id : <code>2062243</code>. You will need to join <code>user</code> and <code>user_artist</code> datasets on the common key <code>artist_id</code>. The sample output is given below.
</div>



In [11]:
def top_n_artists(artist, user_artist, user_id, limit):
    """Returns top n artists liked by a particular user."""
    return (
        user_artist
        .filter(col("user_id") == user_id)
        .join(artist, on="artist_id", how="inner")
        .select("user_id", "playcount", "artist_name")
        .orderBy(col("playcount").desc())
        .limit(limit)
    )

top_n_artists(df_artist, df_user_artist, 2062243, 60).show(truncate=False)


[Stage 7:>                                                          (0 + 1) / 1]

+-------+---------+----------------------+
|user_id|playcount|artist_name           |
+-------+---------+----------------------+
|2062243|26107    |Music 205             |
|2062243|10314    |Mos Def               |
|2062243|7193     |Morrissey             |
|2062243|6652     |Modest Mouse          |
|2062243|4913     |Mouse on Mars         |
|2062243|3983     |The Movielife         |
|2062243|3658     |The Beatles           |
|2062243|3354     |Led Zeppelin          |
|2062243|2843     |Mogwai                |
|2062243|1888     |Queen                 |
|2062243|1718     |Radiohead             |
|2062243|1706     |Motion City Soundtrack|
|2062243|1700     |Talib Kweli           |
|2062243|1470     |Mudvayne              |
|2062243|1359     |Kanye West            |
|2062243|1348     |Jackson Browne        |
|2062243|1257     |Bob Dylan             |
|2062243|1257     |moe.                  |
|2062243|1147     |David Bowie           |
|2062243|1079     |The Killers           |
+-------+--

Here we want to convert data types to integer type where required.

In [12]:
#Cast the data column into integer types
for col_name in df_user_artist.columns:
    df_user_artist = df_user_artist.withColumn(col_name, df_user_artist[col_name].cast(IntegerType()))

df_artist = df_artist.withColumn('artist_id', df_artist['artist_id'].cast(IntegerType()))

In [13]:
df_user_artist.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- artist_id: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [14]:
df_artist.printSchema()

root
 |-- artist_id: integer (nullable = true)
 |-- artist_name: string (nullable = true)



## Train Test Split <a class="anchor" name="train-test-split"></a>

<a class="anchor" id="lab-task-5"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">5. Lab Task: </strong> 
Create training and testing dataset with a 80/20 split
</div>

In [15]:
train, test = df_user_artist.randomSplit([0.8, 0.2], seed=42)


## Model Building <a href="https://spark.apache.org/docs/latest/ml-collaborative-filtering.html" target="_blank">[REF]</a> <a class="anchor" name="model-building"></a>
Collaborative filtering is commonly used for recommender systems. These techniques aim to fill in the missing entries of a user-item association matrix. spark.ml currently supports model-based collaborative filtering, in which users and products are described by a small set of latent factors that can be used to predict missing entries. spark.ml uses the alternating least squares (ALS) algorithm to learn these latent factors. The implementation in spark.ml has the following parameters:

- <strong>numBlocks</strong> is the number of blocks the users and items will be partitioned into in order to parallelize computation (defaults to 10).
- <strong>rank</strong> is the number of latent factors in the model (defaults to 10).
- <strong>maxIter</strong> is the maximum number of iterations to run (defaults to 10).
- <strong>regParam</strong> specifies the regularization parameter in ALS (defaults to 1.0).
- <strong>implicitPrefs</strong> specifies whether to use the explicit feedback ALS variant or one adapted for implicit feedback data (defaults to false which means using explicit feedback).
- <strong>alpha</strong> is a parameter applicable to the implicit feedback variant of ALS that governs the baseline confidence in preference observations (defaults to 1.0).
- <strong>nonnegative</strong> specifies whether or not to use nonnegative constraints for least squares (defaults to false).

In [16]:
als = ALS(maxIter=5, implicitPrefs=True, alpha=40,userCol="user_id", itemCol="artist_id", ratingCol="playcount",
          coldStartStrategy="drop")

<a class="anchor" id="lab-task-6"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">6. Lab Task: </strong> 
Perform the following tasks.
    <ul><li>Train the model with the training set created from above.</li><li> Then transform use the test data to get the predictions. </li><li>Display the first 20 predictions from the results.</li></ul>    
<i>The predictions shown below will be just indicator of how closely a given artist will be to the user's existing preferences</i>
</div>

In [17]:
model = als.fit(train)
predictions = model.transform(test)
predictions.show(20, truncate=False)


+-------+---------+---------+-----------+
|user_id|artist_id|playcount|prediction |
+-------+---------+---------+-----------+
|2007381|83       |74       |2.2862408  |
|2007381|100      |79       |0.8437405  |
|2007381|275      |82       |0.5085737  |
|2007381|1186     |42       |0.34688634 |
|2007381|1198     |324      |0.7884703  |
|2007381|1230     |65       |0.5290074  |
|2007381|1274     |1170     |0.8355888  |
|2007381|1406     |94       |-0.3363917 |
|2007381|1890     |201      |0.8425712  |
|2007381|2717     |101      |1.3816863  |
|2007381|3328     |56       |-1.0661856 |
|2007381|3950     |46       |0.4028274  |
|2007381|4100     |99       |-0.3442504 |
|2007381|4241     |105      |-0.45196462|
|2007381|5043     |89       |0.13683149 |
|2007381|5668     |81       |2.0603185  |
|2007381|1000010  |805      |0.05157852 |
|2007381|1000024  |73       |1.9801524  |
|2007381|1000199  |175      |1.728018   |
|2007381|1000698  |216      |0.9882073  |
+-------+---------+---------+-----

## Evalutation of ALS <a class="anchor" name="evaluation"></a>
We can evaluate ALS using RMSE (Root Mean Squared Error) using the RegressionEvaluator as shown below:

In [18]:
#Write your code here
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(metricName="rmse", labelCol="playcount",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

Root-mean-square error = 1935.7364326163784


<strong style="color:red">NOTE: </strong>If you run the above code, the RMSE you will observe is very high. 

For implicit data, RMSE is not a reliable score since, we don't have any reliable feedback over if items are disliked. RMSE requires knowing which items the user dislikes. Spark does not have a readily available solution for to evaluate the implicit data. The following function implements ROEM (Rank Ordering Error Metric) on the prediction data. You can refer to the details about this <a href="https://campus.datacamp.com/courses/recommendation-engines-in-pyspark/what-if-you-dont-have-customer-ratings?ex=6" target="_BLANK">here</a>.

In [19]:
def ROEM(predictions, userCol="userId", itemCol="songId", ratingCol="num_plays"):
    predictions.createOrReplaceTempView("predictions")
    denominator = predictions.groupBy().sum(ratingCol).collect()[0][0]
    if denominator is None or denominator == 0:
        return float("nan")

    spark.sql(
        "SELECT " + userCol + ", " + ratingCol +
        ", PERCENT_RANK() OVER (PARTITION BY " + userCol +
        " ORDER BY prediction DESC) AS rank FROM predictions"
    ).createOrReplaceTempView("rankings")

    numerator = spark.sql(
        "SELECT SUM(" + ratingCol + " * rank) FROM rankings"
    ).collect()[0][0]

    return numerator / denominator


In [20]:
ROEM(predictions,'user_id','artist_id','playcount')

0.4595136454374456

## Hyperparameter tuning and cross validation <a class="anchor" name="cv"></a>

Since we can't use RMSE as the evaluation metric for "implicit data", we need to manually implement the hyperparameter tuning for the ALS. This code is adapted from the following source [<a href="https://github.com/jamenlong/ALS_expected_percent_rank_cv/blob/master/ROEM_cv.py" target="_BLANK">ref</a>]

<code>alpha</code> is an important hyper-parameter for ALS with implicit feedback. It governs the baseline confidence in preference observations. It is a way to assign a confidence values to the <code>playcount</code>. Higher <code>playcount</code> would mean that we have higher confidence that the user likes that artist and lower <code>playcount</code> would mean the user doesn't like that much.

In [21]:
def ROEM(predictions, userCol="userId", itemCol="songId", ratingCol="num_plays"):
    predictions.createOrReplaceTempView("predictions")
    denominator = predictions.groupBy().sum(ratingCol).collect()[0][0]
    if denominator is None or denominator == 0:
        return float("nan")

    spark.sql(
        "SELECT " + userCol + ", " + ratingCol +
        ", PERCENT_RANK() OVER (PARTITION BY " + userCol +
        " ORDER BY prediction DESC) AS rank FROM predictions"
    ).createOrReplaceTempView("rankings")

    numerator = spark.sql(
        "SELECT SUM(" + ratingCol + " * rank) FROM rankings"
    ).collect()[0][0]

    return numerator / denominator


In [25]:
def ROEM_cv(
    df,
    userCol="user_id",
    itemCol="artist_id",
    ratingCol="playcount",
    ranks=[10],
    maxIters=[10],
    regParams=[0.05],
    alphas=[10, 40]
):
    from pyspark.ml.recommendation import ALS
    from pyspark.sql.functions import rand

    ratings_df = df.orderBy(rand())

    # Train/validation split
    train, validate = ratings_df.randomSplit([0.8, 0.2], seed=0)

    # 3-fold split
    test1, test2, test3 = train.randomSplit([0.33, 0.33, 0.34], seed=1)

    train1 = test2.union(test3)
    train2 = test1.union(test3)
    train3 = test1.union(test2)

    best_validation_performance = float("inf")
    best_rank = 0
    best_maxIter = 0
    best_regParam = 0
    best_alpha = 0
    best_model = None
    best_predictions = None

    for r in ranks:
        for mi in maxIters:
            for rp in regParams:
                for a in alphas:

                    als = ALS(
                        rank=r,
                        maxIter=mi,
                        regParam=rp,
                        alpha=a,
                        userCol=userCol,
                        itemCol=itemCol,
                        ratingCol=ratingCol,
                        coldStartStrategy="drop",
                        nonnegative=True,
                        implicitPrefs=True
                    )

                    # Train the three folds
                    model1 = als.fit(train1)
                    model2 = als.fit(train2)
                    model3 = als.fit(train3)

                    # Predictions
                    predictions1 = model1.transform(test1)
                    predictions2 = model2.transform(test2)
                    predictions3 = model3.transform(test3)

                    def roem_for_predictions(predictions):

                        denominator = (
                            predictions
                            .groupBy()
                            .sum(ratingCol)
                            .collect()[0][0]
                        )

                        if denominator is None or denominator == 0:
                            return float("inf")

                        predictions.createOrReplaceTempView("cv_predictions")

                        spark.sql(
                            "SELECT " + userCol + ", " + ratingCol +
                            ", PERCENT_RANK() OVER "
                            "(PARTITION BY " + userCol +
                            " ORDER BY prediction DESC) AS rank "
                            "FROM cv_predictions"
                        ).createOrReplaceTempView("cv_rankings")

                        numerator = spark.sql(
                            "SELECT SUM(" + ratingCol +
                            " * rank) FROM cv_rankings"
                        ).collect()[0][0]

                        return numerator / denominator

                    performance1 = roem_for_predictions(predictions1)
                    performance2 = roem_for_predictions(predictions2)
                    performance3 = roem_for_predictions(predictions3)

                    print(
                        "Model Parameters:",
                        "\nRank:", r,
                        "\nMaxIter:", mi,
                        "\nRegParam:", rp,
                        "\nAlpha:", a
                    )

                    print(
                        "Test Percent Rank Errors:",
                        performance1,
                        performance2,
                        performance3
                    )

                    # Validate
                    validation_model = als.fit(train)
                    validation_predictions = validation_model.transform(validate)

                    validation_performance = roem_for_predictions(
                        validation_predictions
                    )

                    print(
                        "Validation Percent Rank Error:",
                        validation_performance
                    )

                    if validation_performance < best_validation_performance:

                        best_validation_performance = validation_performance
                        best_rank = r
                        best_maxIter = mi
                        best_regParam = rp
                        best_alpha = a
                        best_model = validation_model
                        best_predictions = validation_predictions

    print("**Best Model**")
    print("Percent Rank Error:", best_validation_performance)
    print("Rank:", best_rank)
    print("MaxIter:", best_maxIter)
    print("RegParam:", best_regParam)
    print("Alpha:", best_alpha)

    return best_model, best_predictions

In [26]:
ROEM_cv(df_user_artist)

Model Parameters: 
Rank: 10 
MaxIter: 10 
RegParam: 0.05 
Alpha: 10
Test Percent Rank Errors: 0.2310596968897655 0.2965007271563484 0.3589835570212632
Validation Percent Rank Error: 0.37235284501274013
Model Parameters: 
Rank: 10 
MaxIter: 10 
RegParam: 0.05 
Alpha: 40
Test Percent Rank Errors: 0.2702442625405783 0.40453195947132914 0.39356297790194406
Validation Percent Rank Error: 0.39543474839361037
**Best Model**
Percent Rank Error: 0.37235284501274013
Rank: 10
MaxIter: 10
RegParam: 0.05
Alpha: 10


(ALSModel: uid=ALS_102625bb7d28, rank=10,
 DataFrame[user_id: int, artist_id: int, playcount: int, prediction: float])

## Making Predictions <a class="anchor" name="predictions"></a>
The k-fold validation implement might take long time to run. You can use the initial ALS model to make the predictions.
Assuming you have successfully trained the model, we want to now use the model to <strong>find top Artists recommended for each user</strong>. We can use the <i><strong>recommendForAllUsers</strong></i> function available in the ALS model to get the list of top recommendations for each users. You can further explore the details of the API <a href="https://spark.apache.org/docs/2.2.0/api/python/pyspark.ml.html#pyspark.ml.recommendation.ALS" target="_blank">here</a>.

The <code>recommendForAllUsers</code> only gives the list of artist_ids for the users, you can write the code to map these artist_ids back to their names.

In [27]:
model.recommendForAllUsers(10).show(truncate=False)

+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                                      |
+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1001440|[{1000123, 5.415}, {1, 4.4043355}, {1004162, 3.930635}, {4163, 3.1794863}, {1200, 3.1124835}, {1238230, 3.0878634}, {437, 2.7519276}, {1002734, 2.7283378}, {1000010, 2.6772795}, {703, 2.64886}]                    |
|1021940|[{1001169, 4.800869}, {1001907, 4.5999804}, {1169, 4.5403576}, {1000495, 4.006379}, {4499, 3.86

<a class="anchor" id="lab-task-7"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">7. Lab Task: </strong> 
    Write a function to find the top <strong>N</strong> recommended artists for the user : <strong>2062243</strong>. Display  <code>artist_id and artist_name</code> both. A sample output is given below.
</div>

![image.png](attachment:image.png)

In [28]:
def recommendedArtists(als_model, user_id, limit):
    selected_user = (
        als_model
        .recommendForAllUsers(limit)
        .filter(col("user_id") == user_id)
    )

    top_artist = (
        selected_user
        .selectExpr("user_id", "explode(recommendations) as recommendation")
        .select(
            "user_id",
            col("recommendation.artist_id").alias("artist_id"),
            col("recommendation.rating").alias("prediction")
        )
    )

    final = (
        top_artist
        .join(df_artist, on="artist_id", how="left")
        .select("user_id", "artist_id", "artist_name", "prediction")
        .orderBy(col("prediction").desc())
    )
    return final


In [29]:
recommendedArtists(model, 2062243, 20).show(truncate=False)


+-------+---------+------------------------+----------+
|user_id|artist_id|artist_name             |prediction|
+-------+---------+------------------------+----------+
|2062243|1004162  |In Flames               |7.266914  |
|2062243|1000323  |Guns N' Roses           |5.2303777 |
|2062243|1002128  |Taking Back Sunday      |5.199199  |
|2062243|1238230  |Straylight Run          |5.1588078 |
|2062243|1002734  |Catch 22                |4.2997527 |
|2062243|4163     |Britney Spears          |4.0133996 |
|2062243|1000445  |Hoobastank              |3.773422  |
|2062243|1002551  |Avenged Sevenfold       |3.7009814 |
|2062243|742      |Placebo                 |3.4938917 |
|2062243|1004278  |Brand New               |3.2661047 |
|2062243|1200     |Faith No More           |3.1383362 |
|2062243|4225     |Electric Six            |3.0701678 |
|2062243|1014421  |Rage Against the Machine|2.9205287 |
|2062243|1278     |Ryan Adams              |2.7323844 |
|2062243|703      |Janet Jackson           |2.67

### Congratulations on finishing this activity. See you next week.